# Pb2MgWO6

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib widget

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize

In [ ]:
from util.io import load_foldslice
from util.filter import remove_linear_ramp

obj, obj_sampling = load_foldslice("data/PMW111.mat")
phase = remove_linear_ramp(np.angle(obj))
phase -= np.nanquantile(phase, 0.1)

print(f"type(obj): {type(obj)}")
print(f"obj dtype: {obj.dtype} shape: {obj.shape}")

In [ ]:
contrast = np.std(phase, axis=(-1, -2))

fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
ax.set_ylabel("Contrast")
ax.set_xlabel("Slice #")

ax.bar(np.arange(len(phase)), contrast, width=0.9)

plt.show()

In [ ]:
phase_crop = phase[14:34]
obj_crop = obj[14:34]

mean_phase = np.mean(phase_crop, axis=0)
mean_int = np.mean(np.abs(obj_crop)**2, axis=0)

fig, axs = plt.subplots(ncols=2, sharex=True, sharey=True, constrained_layout=True)
fig.set_size_inches(6, 3)

fig.suptitle("Mean image")
fig.colorbar(axs[0].imshow(mean_phase), label="Mean phase [rad]", shrink=0.7)
fig.colorbar(axs[1].imshow(mean_int, vmax=1.0), label="Mean intensity", shrink=0.7)
plt.show()

We find the peaks and index as before

In [ ]:
from util.peaks import find_peaks, integrate_peaks

template_sigma = 1.5  # std. deviation of Gaussian template (px)
threshold_corr = 0.7  # threshold normalized cross correlation to consider it a peak

# check the definition of this function to see the details
(cross_corr, peaks) = find_peaks(mean_phase, template_sigma, threshold_corr, min_distance=3)

peak_ints = integrate_peaks(mean_phase, peaks, 0.7 / obj_sampling[0])

fig, axs = plt.subplots(ncols=2, constrained_layout=True, gridspec_kw={'width_ratios': [4, 1]})
axs[0].set_axis_off()
axs[1].set_ylabel("Mean Phase @ peak")

vmin, vmax = float(np.min(peak_ints)), float(np.max(peak_ints))

cmap = plt.get_cmap('magma')
norm = Normalize(vmin, vmax)

axs[0].imshow(mean_phase, cmap='gray')
axs[0].scatter(peaks[:, 1], peaks[:, 0], s=2, c=peak_ints, cmap=cmap, norm=norm)

hist, bins = np.histogram(peak_ints, 50, range=(vmin, vmax))
bin_heights = bins[1:] - bins[:-1]  # type: ignore
axs[1].barh(bins[:-1], hist, height=bin_heights, align='edge', color=cmap(norm(bins[:-1])))

plt.show()

In [ ]:
threshold = 4.0

sel_peaks = peaks[peak_ints > threshold]
sel_peak_ints = peak_ints[peak_ints > threshold]
print(f"sel_peaks: {len(sel_peaks)}/{len(peaks)}")

fig, ax = plt.subplots()
ax.set_axis_off()

ax.imshow(mean_phase, cmap='gray')
ax.scatter(sel_peaks[:, 1], sel_peaks[:, 0], s=2, c=sel_peak_ints, cmap=cmap, norm=norm)

plt.show()

In [ ]:
from util.index import find_lattice_directions
from util.plot import plot_found_lattice_directions

found_dirs = find_lattice_directions(mean_phase, n_peaks=5)

plot_found_lattice_directions(found_dirs);

In [ ]:
from util.index import graph_index

x_dir = 4
y_dir = 0

grid_indices, grid_peaks = graph_index(sel_peaks, found_dirs, x_dir=x_dir, y_dir=y_dir)

fig, axs = plt.subplots(ncols=4, constrained_layout=True)
fig.set_size_inches(8, 3)
[ax.set_axis_off() for ax in axs.flat]

axs[0].imshow(grid_peaks[..., 0])
axs[0].set_title("Y peak position")
axs[1].imshow(grid_peaks[..., 1])
axs[1].set_title("X peak position")

rr, cc = np.indices(grid_indices.shape)

axs[2].imshow(mean_phase, cmap='gray')
axs[2].scatter(grid_peaks[..., 1].ravel(), grid_peaks[..., 0].ravel(), s=5, c=rr.ravel())
axs[2].set_title("Peak Row #")

axs[3].imshow(mean_phase, cmap='gray')
axs[3].scatter(grid_peaks[..., 1].ravel(), grid_peaks[..., 0].ravel(), s=5, c=cc.ravel())
axs[3].set_title("Peak Col #")

plt.show()

We want to filter our grid to only contain the Pb atoms. We can do this by slicing:

In [ ]:
grid_peaks = grid_peaks[1::4, 1::2]  # take every other row & col

fig, axs = plt.subplots(ncols=2, constrained_layout=True)
fig.set_size_inches(4, 3)
[ax.set_axis_off() for ax in axs.flat]

axs[0].imshow(grid_peaks[..., 0])
axs[0].set_title("Y peak position")
axs[1].imshow(grid_peaks[..., 1])
axs[1].set_title("X peak position")

plt.show()

Now our unit cell is the double perovskite unit cell. We can select positions:

In [ ]:
from util.index import construct_grid_points

pb_peaks = construct_grid_points(grid_peaks, [[0., 0.], [0.0, 0.5], [0.5, 0.0], [0.5, 0.5]])
b1_peaks = construct_grid_points(grid_peaks, [[0.25, 0.0], [0.75, 0.5]])
b2_peaks = construct_grid_points(grid_peaks, [[0.25, 0.5], [0.75, 0.0]])
o_peaks = construct_grid_points(grid_peaks, [[0.25, 0.25], [0.25, 0.75], [0.75, 0.25], [0.75, 0.75]])

fig, ax = plt.subplots(constrained_layout=True)
ax.set_axis_off()

s = 10
ax.scatter(pb_peaks[:, 1], pb_peaks[:, 0], s=s, c='gray', label="Pb")
ax.scatter(b1_peaks[:, 1], b1_peaks[:, 0], s=s, c='blue', label="B1")
ax.scatter(b2_peaks[:, 1], b2_peaks[:, 0], s=s, c='green', label="B2")
ax.scatter(o_peaks[:, 1], o_peaks[:, 0], s=s, c='red', label="O")

ax.imshow(mean_phase)

ax.legend(loc='lower right')
plt.show()

In [ ]:
from util.plot import plot_voronoi

integration_radius = 0.5

b1_peak_ints = integrate_peaks(mean_phase, b1_peaks, integration_radius / obj_sampling[0])
b2_peak_ints = integrate_peaks(mean_phase, b2_peaks, integration_radius / obj_sampling[0])

fig, axs = plt.subplots(ncols=2, sharex=True, sharey=True)

axs[0].set_axis_off()
axs[1].set_axis_off()
axs[0].imshow(mean_phase, cmap='gray')
axs[1].imshow(mean_phase, cmap='gray')

axs[0].set_title("B1 site intensity")
plot_voronoi(axs[0], b1_peaks[..., 1], b1_peaks[..., 0], b1_peak_ints, mean_phase.shape, cmap='magma')
axs[1].set_title("B2 site intensity")
plot_voronoi(axs[1], b2_peaks[..., 1], b2_peaks[..., 0], b2_peak_ints, mean_phase.shape, cmap='magma')

plt.show()

The phase boundary looks diffuse. But let's check the slices:

In [ ]:
selected_slices = [0, 5, 10, 15, 19]

integration_radius = 0.5  # angstrom

fig, axs = plt.subplots(ncols=len(selected_slices), sharex=True, sharey=True, constrained_layout=True)
fig.set_size_inches(8, 3)

for (slice_i, ax) in zip(selected_slices, axs.flat):
    ax.set_axis_off()
    ax.imshow(phase_crop[slice_i], cmap='gray')

    diff = (
        integrate_peaks(phase_crop[slice_i], b1_peaks, integration_radius / obj_sampling[0]) - 
        integrate_peaks(phase_crop[slice_i], b2_peaks, integration_radius / obj_sampling[0])
    )

    sm = plot_voronoi(ax, b1_peaks[..., 1], b1_peaks[..., 0], diff, mean_phase.shape, cmap='bwr', vmin=-5, vmax=5)

fig.colorbar(sm, ax=axs, label="B1 - B2 intensity")

plt.show()

### Animation

In [ ]:
from matplotlib.animation import FuncAnimation
from ipywidgets import HTML

plt.close('all')

fig, axs = plt.subplots(ncols=2, constrained_layout=True, sharex=True, sharey=True)
fig.set_size_inches(8, 4)

axs[0].set_axis_off() 
axs[1].set_axis_off()
axs[1].set_title("B-site contrast")

layer = 0

img_vmin, img_vmax = float(np.nanmin(phase_crop)), float(np.nanmax(phase_crop))
img = axs[0].imshow(phase_crop[layer], cmap='gray', vmin=img_vmin, vmax=img_vmax)

vmin, vmax = -5, 5

diff = (
    integrate_peaks(phase_crop[layer], b1_peaks, integration_radius / obj_sampling[0]) - 
    integrate_peaks(phase_crop[layer], b2_peaks, integration_radius / obj_sampling[0])
)
quadmesh = plot_voronoi(axs[1], b1_peaks[..., 1], b1_peaks[..., 0], diff, mean_phase.shape,
                        cmap='magma', vmin=vmin, vmax=vmax, max_r=integration_radius / obj_sampling[0])

text = axs[0].text(0.03, 0.97, f"layer: {layer:2}", ha='left', va='top', transform=axs[0].transAxes, color='red')

def animate(layer: int):
    diff = (
        integrate_peaks(phase_crop[layer], b1_peaks, integration_radius / obj_sampling[0]) - 
        integrate_peaks(phase_crop[layer], b2_peaks, integration_radius / obj_sampling[0])
    )

    try:
        global quadmesh
        if quadmesh is not None:
            try:
                quadmesh.remove()
            except ValueError:
                pass
        quadmesh = plot_voronoi(axs[1], b1_peaks[..., 1], b1_peaks[..., 0], diff, mean_phase.shape,
                                cmap='magma', vmin=vmin, vmax=vmax)

        img.set_data(phase_crop[layer])
        text.set_text(f"layer: {layer:2}")
    except NameError:
        pass
    return ()

fig.draw_without_rendering()
anim = FuncAnimation(fig, animate, frames=len(phase_crop), interval=1000)

anim.save("bsite_contrast.mp4", dpi=300, fps=3)
#display(HTML(anim.to_jshtml(fps=3)))

anim.event_source.stop()
plt.close(fig)
del animate, anim, img, quadmesh, text

## Strain analysis

Let's measure the Pb-Pb bond distance. We'll need 2d Gaussian peak fitting to get accurate peak positions

In [ ]:
from util.peaks import fit_2d_peaks_masked
from util.index import construct_grid_points_masked

pb_peaks = construct_grid_points_masked(grid_peaks, [[0., 0.], [0.0, 0.5], [0.5, 0.0], [0.5, 0.5]])

pb_peak_fit, residuals = fit_2d_peaks_masked(mean_phase, pb_peaks, cutout_r=8, sigma=1., use_bg=True)

fig, ax = plt.subplots()

ax.imshow(residuals)
plt.show()

Now we can calculate the bond distances:

In [ ]:
dists = np.concatenate((
    np.sqrt((pb_peak_fit[1]['x'] - pb_peak_fit[0]['x'])**2 + (pb_peak_fit[1]['y'] - pb_peak_fit[0]['y'])**2).ravel(),
    np.sqrt((pb_peak_fit[0, :, 1:]['x'] - pb_peak_fit[1, :, :-1]['x'])**2 + (pb_peak_fit[0, :, 1:]['y'] - pb_peak_fit[1, :, :-1]['y'])**2).ravel(),

    np.sqrt((pb_peak_fit[3]['x'] - pb_peak_fit[2]['x'])**2 + (pb_peak_fit[3]['y'] - pb_peak_fit[2]['y'])**2).ravel(),
    np.sqrt((pb_peak_fit[2, :, 1:]['x'] - pb_peak_fit[3, :, :-1]['x'])**2 + (pb_peak_fit[2, :, 1:]['y'] - pb_peak_fit[3, :, :-1]['y'])**2).ravel(),
))
dists *= obj_sampling[0] * 100  # pm

xs = np.concatenate((pb_peak_fit[0]['x'].ravel(), pb_peak_fit[1, :, :-1]['x'].ravel(), pb_peak_fit[2]['x'].ravel(), pb_peak_fit[3, :, :-1]['x'].ravel()))
ys = np.concatenate((pb_peak_fit[0]['y'].ravel(), pb_peak_fit[1, :, :-1]['y'].ravel(), pb_peak_fit[2]['y'].ravel(), pb_peak_fit[3, :, :-1]['y'].ravel()))

mask = np.isfinite(xs.data) & np.isfinite(dists.data)
dists = dists.data[mask]
ys = ys.data[mask]
xs = xs.data[mask]

fig, ax = plt.subplots()

sm = plot_voronoi(ax, xs, ys, dists, mean_phase.shape, cmap='bwr')
fig.colorbar(sm, label="Pb-Pb dist [pm]")

plt.show()